# Fraud-Spike Detector — Merchant Behavioral Risk Scoring
### Razorpay AI Buildathon — Track 02: AI Risk Manager

**Problem statement:** Detect when a merchant's transaction behavior deviates anomalously — in *shape*, not just volume — from their own historical pattern, distinguishing genuine demand-driven spikes (e.g. festivals) from anomalous ones, and produce a confidence-scored anomaly signal evaluated with precision, recall, and false-positive cost on unseen data.

**Dataset:** Kartik2112 Credit Card Transactions Fraud Detection (Kaggle) — simulated Sparkov transaction data. `fraudTrain.csv` (Jan 2019–Jun 2020) is used purely as historical context; `fraudTest.csv` (Jun–Dec 2020) is the unseen evaluation period. 693 merchants appear in both, with a clean temporal boundary between them.

**Core design decision:** The unit of analysis is a **merchant-day**, not a single transaction — because a "spike" is inherently a pattern over time, not a property of one transaction. For each merchant-day, we compare that day's behavior to the *same merchant's own historical baseline*, since a normal day for a large merchant looks nothing like a normal day for a small one.


## Step -1 — Install dependencies
Run this once per environment/session.

In [ ]:
!pip install xgboost -q
from IPython.display import display

## Step 0 — Load the raw data
Upload `fraudTrain.csv` and `fraudTest.csv` to this notebook's working directory before running.

In [ ]:
import pandas as pd

# Load both files
train = pd.read_csv('fraudTrain.csv')
test = pd.read_csv('fraudTest.csv')

print("TRAIN shape:", train.shape)
print("TEST shape:", test.shape)

print("\n--- Columns ---")
print(train.columns.tolist())

print("\n--- First 3 rows ---")
print(train.head(3))

print("\n--- Data types ---")
print(train.dtypes)

print("\n--- Missing values ---")
print(train.isnull().sum())

print("\n--- Fraud class balance (train) ---")
print(train['is_fraud'].value_counts())
print(train['is_fraud'].value_counts(normalize=True))

print("\n--- Number of unique merchants ---")
print(train['merchant'].nunique())

print("\n--- Transactions per merchant (summary stats) ---")
print(train.groupby('merchant').size().describe())

print("\n--- Date range ---")
print(train['trans_date_trans_time'].min(), "to", train['trans_date_trans_time'].max())

## Step 1 — Clean & confirm the train/test structure
Confirms: a genuine temporal split (train ends seconds before test begins), all 693 merchants present in both, and fraud spread across most merchants rather than concentrated in a few — meaning this is a diffuse behavioral pattern, not a small ring of obviously-bad actors (which would need network/graph analysis instead).

In [ ]:
import pandas as pd

# --- Clean & type-fix ---
train['trans_date_trans_time'] = pd.to_datetime(train['trans_date_trans_time'])
test['trans_date_trans_time'] = pd.to_datetime(test['trans_date_trans_time'])

# --- Check date ranges ---
print("TRAIN date range:", train['trans_date_trans_time'].min(), "to", train['trans_date_trans_time'].max())
print("TEST date range:", test['trans_date_trans_time'].min(), "to", test['trans_date_trans_time'].max())

# --- Check merchant overlap between train and test ---
train_merchants = set(train['merchant'].unique())
test_merchants = set(test['merchant'].unique())
print("\nMerchants in TRAIN:", len(train_merchants))
print("Merchants in TEST:", len(test_merchants))
print("Merchants in BOTH:", len(train_merchants & test_merchants))
print("Merchants ONLY in TEST (no history):", len(test_merchants - train_merchants))

# --- Fraud distribution across merchants (train) ---
fraud_per_merchant = train.groupby('merchant')['is_fraud'].sum().sort_values(ascending=False)
print("\nMerchants with at least 1 fraud txn (train):", (fraud_per_merchant > 0).sum(), "out of", len(fraud_per_merchant))
print("Top 10 merchants by fraud count:\n", fraud_per_merchant.head(10))

# --- Transactions per merchant per day (train), to gauge window granularity ---
train['date'] = train['trans_date_trans_time'].dt.date
per_merchant_per_day = train.groupby(['merchant', 'date']).size()
print("\nTransactions per merchant per active day - summary:")
print(per_merchant_per_day.describe())

# --- How many days does each merchant appear on? ---
active_days_per_merchant = train.groupby('merchant')['date'].nunique()
print("\nActive days per merchant - summary:")
print(active_days_per_merchant.describe())

## Step 1b — Build the merchant-day table for TRAIN
Before we can compute walk-forward baselines (Step 2), we need one row per merchant per day, aggregated from raw transactions: transaction count, total amount, average amount, amount volatility, unique customers, and active hours — plus whether that day contained any fraud (used only for evaluation later, never as a model input).

In [ ]:
# --- Ensure hour-of-day is available (used for the "active hours" feature) ---
train['hour'] = train['trans_date_trans_time'].dt.hour

# --- Sort chronologically per merchant (required before any walk-forward/cumulative stat) ---
train = train.sort_values(['merchant', 'trans_date_trans_time'])

# --- Aggregate raw transactions into one row per merchant per day ---
daily_train = (
    train.groupby(['merchant', 'date'])
    .agg(
        txn_count=('amt', 'count'),
        total_amt=('amt', 'sum'),
        avg_amt=('amt', 'mean'),
        std_amt=('amt', 'std'),
        unique_customers=('cc_num', 'nunique'),
        active_hours=('hour', 'nunique'),
        had_fraud=('is_fraud', 'max')
    )
    .reset_index()
)

# A day with only one transaction has an undefined standard deviation -> treat as 0 (no variability)
daily_train['std_amt'] = daily_train['std_amt'].fillna(0)

print("Training merchant-days:", len(daily_train))
print(daily_train.head())


## Step 2 — Build leakage-safe, walk-forward merchant baselines (TRAIN)

**Why this is the most important cell in the notebook.** An earlier version of this pipeline computed each merchant's "normal" behavior using their *entire* history — including days that hadn't happened yet relative to the day being scored. That's look-ahead bias: it lets the model see the future when judging the past, and it makes evaluation metrics look better than they would in a real, live system.

The fix: for every merchant-day, the baseline (mean/variance of transaction count, total amount, average amount, amount volatility, unique customers, and active hours) is built **only from that merchant's days strictly before it** — using a cumulative sum shifted by one day (`shift(1)`), so day *N* never sees day *N* or any day after it. This mirrors exactly what a real fraud system would know if it were running live on that date.

Each day's numbers are converted to a **z-score**: how many standard deviations away from *this merchant's own normal* today's behavior sits. This makes merchants comparable on the same scale regardless of their size.

In [ ]:
# ============================================================
# PHASE 9 — STRICT PAST-ONLY TRAINING FEATURES
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Make sure training data is sorted chronologically
# ------------------------------------------------------------

daily_train = daily_train.sort_values(
    ["merchant", "date"]
).reset_index(drop=True)


# ------------------------------------------------------------
# 2. Create cumulative historical statistics
# ------------------------------------------------------------
# IMPORTANT:
# shift(1) means today's values are NOT included.
# Therefore every day's baseline uses ONLY previous days.

g = daily_train.groupby("merchant", sort=False)

daily_train["hist_n"] = g.cumcount()

daily_train["hist_sum_count"] = (
    g["txn_count"].cumsum().shift(1)
)

daily_train["hist_sum_count_sq"] = (
    g["txn_count"]
    .apply(lambda x: (x ** 2).cumsum())
    .reset_index(level=0, drop=True)
    .shift(1)
)

daily_train["hist_sum_total_amt"] = (
    g["total_amt"].cumsum().shift(1)
)

daily_train["hist_sum_total_amt_sq"] = (
    g["total_amt"]
    .apply(lambda x: (x ** 2).cumsum())
    .reset_index(level=0, drop=True)
    .shift(1)
)

daily_train["hist_sum_avg_amt"] = (
    g["avg_amt"].cumsum().shift(1)
)

daily_train["hist_sum_avg_amt_sq"] = (
    g["avg_amt"]
    .apply(lambda x: (x ** 2).cumsum())
    .reset_index(level=0, drop=True)
    .shift(1)
)

daily_train["hist_sum_std_amt"] = (
    g["std_amt"].cumsum().shift(1)
)

daily_train["hist_sum_std_amt_sq"] = (
    g["std_amt"]
    .apply(lambda x: (x ** 2).cumsum())
    .reset_index(level=0, drop=True)
    .shift(1)
)

daily_train["hist_sum_customers"] = (
    g["unique_customers"].cumsum().shift(1)
)

daily_train["hist_sum_customers_sq"] = (
    g["unique_customers"]
    .apply(lambda x: (x ** 2).cumsum())
    .reset_index(level=0, drop=True)
    .shift(1)
)

daily_train["hist_sum_hours"] = (
    g["active_hours"].cumsum().shift(1)
)

daily_train["hist_sum_hours_sq"] = (
    g["active_hours"]
    .apply(lambda x: (x ** 2).cumsum())
    .reset_index(level=0, drop=True)
    .shift(1)
)


# ------------------------------------------------------------
# 3. Function to calculate historical z-score
# ------------------------------------------------------------

def historical_z(value, hist_sum, hist_sum_sq, n):

    # No previous history
    if n < 2:
        return 0.0

    mean = hist_sum / n

    variance = (
        hist_sum_sq / n
    ) - (mean ** 2)

    variance = max(variance, 0)

    std = np.sqrt(variance)

    if std == 0 or np.isnan(std):
        std = 1.0

    return (value - mean) / std


# ------------------------------------------------------------
# 4. Calculate the six deviation features
# ------------------------------------------------------------

daily_train["z_count"] = daily_train.apply(
    lambda r: historical_z(
        r["txn_count"],
        r["hist_sum_count"],
        r["hist_sum_count_sq"],
        r["hist_n"]
    ),
    axis=1
)

daily_train["z_total_amt"] = daily_train.apply(
    lambda r: historical_z(
        r["total_amt"],
        r["hist_sum_total_amt"],
        r["hist_sum_total_amt_sq"],
        r["hist_n"]
    ),
    axis=1
)

daily_train["z_avg_amt"] = daily_train.apply(
    lambda r: historical_z(
        r["avg_amt"],
        r["hist_sum_avg_amt"],
        r["hist_sum_avg_amt_sq"],
        r["hist_n"]
    ),
    axis=1
)

daily_train["z_std_amt"] = daily_train.apply(
    lambda r: historical_z(
        r["std_amt"],
        r["hist_sum_std_amt"],
        r["hist_sum_std_amt_sq"],
        r["hist_n"]
    ),
    axis=1
)

daily_train["z_unique_customers"] = daily_train.apply(
    lambda r: historical_z(
        r["unique_customers"],
        r["hist_sum_customers"],
        r["hist_sum_customers_sq"],
        r["hist_n"]
    ),
    axis=1
)

daily_train["z_active_hours"] = daily_train.apply(
    lambda r: historical_z(
        r["active_hours"],
        r["hist_sum_hours"],
        r["hist_sum_hours_sq"],
        r["hist_n"]
    ),
    axis=1
)


# ------------------------------------------------------------
# 5. Clean the deviation features
# ------------------------------------------------------------

features = [
    "z_count",
    "z_total_amt",
    "z_avg_amt",
    "z_std_amt",
    "z_unique_customers",
    "z_active_hours"
]

daily_train[features] = (
    daily_train[features]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)


# ------------------------------------------------------------
# 6. Create the final training dataset
# ------------------------------------------------------------

train_model_strict = daily_train[
    ["merchant", "date", "had_fraud"] + features
].copy()


# ------------------------------------------------------------
# 7. Check results
# ------------------------------------------------------------

print("Strict training dataset:")
print(train_model_strict.shape)

print("\nFraud-day distribution:")
print(
    train_model_strict["had_fraud"].value_counts()
)

print("\nFraud-day percentage:")
print(
    f"{train_model_strict['had_fraud'].mean() * 100:.2f}%"
)

print("\nAverage deviations by fraud label:")

display(
    train_model_strict.groupby("had_fraud")[features].mean()
)

print("\nSample:")
display(
    train_model_strict.head(10)
)

## Step 3 — Train the supervised model (XGBoost) on leakage-safe features
Trained on a temporal 80/20 split (never randomly shuffled — later days are held out, mimicking a real deployment where you only ever have the past to train on). Class imbalance (~2% fraud-days) is handled via `scale_pos_weight` rather than resampling, which keeps the real class distribution intact for honest evaluation.

In [ ]:
# ============================================================
# PHASE 10 — RETRAIN XGBOOST WITH STRICT PAST-ONLY FEATURES
# ============================================================

from xgboost import XGBClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)

# ------------------------------------------------------------
# 1. Sort chronologically
# ------------------------------------------------------------

train_model_strict = train_model_strict.sort_values(
    ["date", "merchant"]
).reset_index(drop=True)


# ------------------------------------------------------------
# 2. Define features and target
# ------------------------------------------------------------

features = [
    "z_count",
    "z_total_amt",
    "z_avg_amt",
    "z_std_amt",
    "z_unique_customers",
    "z_active_hours"
]

X = train_model_strict[features]
y = train_model_strict["had_fraud"]


# ------------------------------------------------------------
# 3. Temporal 80/20 split
# ------------------------------------------------------------

split_point = int(len(train_model_strict) * 0.80)

X_train = X.iloc[:split_point]
y_train = y.iloc[:split_point]

X_val = X.iloc[split_point:]
y_val = y.iloc[split_point:]


print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))

print("\nTraining fraud rate:")
print(f"{y_train.mean() * 100:.2f}%")

print("\nValidation fraud rate:")
print(f"{y_val.mean() * 100:.2f}%")


# ------------------------------------------------------------
# 4. Handle class imbalance
# ------------------------------------------------------------

negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("\nScale positive weight:")
print(round(scale_pos_weight, 2))


# ------------------------------------------------------------
# 5. Train XGBoost
# ------------------------------------------------------------

model_strict = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,

    objective="binary:logistic",

    scale_pos_weight=scale_pos_weight,

    eval_metric="logloss",

    random_state=42,
    n_jobs=-1
)

model_strict.fit(
    X_train,
    y_train
)


# ------------------------------------------------------------
# 6. Get validation probabilities
# ------------------------------------------------------------

val_probability_strict = model_strict.predict_proba(
    X_val
)[:, 1]


# ------------------------------------------------------------
# 7. Evaluate at threshold = 0.50
# ------------------------------------------------------------

threshold = 0.50

val_prediction = (
    val_probability_strict >= threshold
).astype(int)


# ------------------------------------------------------------
# 8. Metrics
# ------------------------------------------------------------

precision = precision_score(
    y_val,
    val_prediction,
    zero_division=0
)

recall = recall_score(
    y_val,
    val_prediction,
    zero_division=0
)

f1 = f1_score(
    y_val,
    val_prediction,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_val,
    val_probability_strict
)

pr_auc = average_precision_score(
    y_val,
    val_probability_strict
)

cm = confusion_matrix(
    y_val,
    val_prediction
)


# ------------------------------------------------------------
# 9. Display results
# ------------------------------------------------------------

print("\n======================================")
print("STRICT TEMPORAL VALIDATION RESULTS")
print("======================================")

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")
print(f"PR-AUC    : {pr_auc:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")

print(
    classification_report(
        y_val,
        val_prediction,
        digits=4,
        zero_division=0
    )
)

## Step 3b — Baseline comparison: Isolation Forest (unsupervised)

Before committing to a supervised approach, we tested an unsupervised **Isolation Forest** on the same deviation features, since it requires no labels and can in principle catch anomaly patterns that don't resemble any labeled fraud example. On this dataset it performed weakly (low precision and recall) compared to the supervised model — reasonable, since we *do* have reliable fraud labels here, and the class imbalance is severe enough that isolating "any statistical outlier" catches far more benign spikes (e.g. a genuinely busy day) than actual fraud.

We report this honestly rather than omitting it: it's evidence of evaluating an alternative, not just defaulting to the fashionable choice. XGBoost is used as the primary model going forward.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score

iso = IsolationForest(n_estimators=200, contamination=0.02, random_state=42)
iso.fit(X_train)

iso_val_flag = (iso.predict(X_val) == -1).astype(int)

print("Isolation Forest (unsupervised) on validation set:")
print(f"Precision: {precision_score(y_val, iso_val_flag, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_val, iso_val_flag, zero_division=0):.4f}")
print("\n(Compare to the supervised XGBoost validation results above.)")


## Step 4 — Threshold tuning
A single 0.5 threshold is arbitrary. We sweep thresholds to see the full precision/recall trade-off, since the right operating point depends on business priorities, not on a default.

In [ ]:
# ============================================================
# PHASE 11 — STRICT MODEL THRESHOLD TUNING
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

thresholds = np.arange(0.10, 0.96, 0.05)

strict_results = []

for threshold in thresholds:

    prediction = (
        val_probability_strict >= threshold
    ).astype(int)

    precision = precision_score(
        y_val,
        prediction,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        prediction,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        prediction,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        prediction
    ).ravel()

    alert_rate = (
        (fp + tp) / len(y_val)
    )

    strict_results.append({
        "threshold": round(threshold, 2),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp,
        "alert_rate": alert_rate
    })


strict_threshold_results = pd.DataFrame(
    strict_results
)

print("STRICT MODEL — THRESHOLD COMPARISON")

display(
    strict_threshold_results.round(4)
)

## Step 5 — Business cost analysis
Precision and recall alone don't say what threshold to actually use — that depends on how costly a missed fraud-day is relative to a false alarm. We model this explicitly across a few plausible cost ratios rather than picking one threshold arbitrarily.

In [ ]:
# ============================================================
# PHASE 12 — FINAL COST ANALYSIS
# ============================================================

FP_COST = 1

FN_COSTS = [5, 10, 20, 50]

final_cost_results = []

for _, row in strict_threshold_results.iterrows():

    result = {
        "threshold": row["threshold"],
        "precision": row["precision"],
        "recall": row["recall"],
        "f1": row["f1"],
        "alert_rate": row["alert_rate"],
        "false_positives": int(row["false_positives"]),
        "false_negatives": int(row["false_negatives"]),
        "true_positives": int(row["true_positives"])
    }

    for fn_cost in FN_COSTS:

        result[f"cost_FN_{fn_cost}"] = (
            row["false_positives"] * FP_COST
            + row["false_negatives"] * fn_cost
        )

    final_cost_results.append(result)


final_cost_df = pd.DataFrame(
    final_cost_results
)


# ------------------------------------------------------------
# Display full cost table
# ------------------------------------------------------------

print("FINAL STRICT-MODEL COST ANALYSIS")

display(
    final_cost_df.round(4)
)


# ------------------------------------------------------------
# Find best threshold for each cost assumption
# ------------------------------------------------------------

print("\n======================================")
print("RECOMMENDED OPERATING THRESHOLDS")
print("======================================")

for fn_cost in FN_COSTS:

    cost_column = f"cost_FN_{fn_cost}"

    best = final_cost_df.loc[
        final_cost_df[cost_column].idxmin()
    ]

    print(
        f"\nFN cost = {fn_cost}x FP"
    )

    print(
        f"Threshold : {best['threshold']:.2f}"
    )

    print(
        f"Precision : {best['precision']:.4f}"
    )

    print(
        f"Recall    : {best['recall']:.4f}"
    )

    print(
        f"F1        : {best['f1']:.4f}"
    )

    print(
        f"Alert rate: {best['alert_rate']:.4f}"
    )

    print(
        f"False positives: {best['false_positives']}"
    )

    print(
        f"False negatives: {best['false_negatives']}"
    )

    print(
        f"Total cost: {int(best[cost_column])}"
    )


## Step 6 — Build the same leakage-safe features for TEST (walk-forward, continuing from TRAIN history)
Test-day features use each merchant's full prior history — their entire train period, plus any earlier test days — never anything from that day forward or beyond. This is the same walk-forward principle as Step 2, extended across the train/test boundary. (This cell rebuilds its own train/test daily aggregates from the raw data, independently of Step 1b, so it can construct the combined train+test walk-forward sequence correctly.)

In [ ]:
# ============================================================
# PHASE 13 — COMPLETE STRICT TEST FEATURE PIPELINE
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Prepare TRAIN transaction data
# ------------------------------------------------------------

train["trans_date_trans_time"] = pd.to_datetime(
    train["trans_date_trans_time"]
)

train["date"] = train["trans_date_trans_time"].dt.date
train["hour"] = train["trans_date_trans_time"].dt.hour

train = train.sort_values(
    ["merchant", "trans_date_trans_time"]
)


# ------------------------------------------------------------
# 2. Prepare TEST transaction data
# ------------------------------------------------------------

test["trans_date_trans_time"] = pd.to_datetime(
    test["trans_date_trans_time"]
)

test["date"] = test["trans_date_trans_time"].dt.date
test["hour"] = test["trans_date_trans_time"].dt.hour

test = test.sort_values(
    ["merchant", "trans_date_trans_time"]
)


# ------------------------------------------------------------
# 3. Create merchant-day TRAIN data
# ------------------------------------------------------------

daily_train_test = (
    train.groupby(["merchant", "date"])
    .agg(
        txn_count=("amt", "count"),
        total_amt=("amt", "sum"),
        avg_amt=("amt", "mean"),
        std_amt=("amt", "std"),
        unique_customers=("cc_num", "nunique"),
        active_hours=("hour", "nunique")
    )
    .reset_index()
)

daily_train_test["std_amt"] = (
    daily_train_test["std_amt"].fillna(0)
)


# ------------------------------------------------------------
# 4. Create merchant-day TEST data
# ------------------------------------------------------------

daily_test = (
    test.groupby(["merchant", "date"])
    .agg(
        txn_count=("amt", "count"),
        total_amt=("amt", "sum"),
        avg_amt=("amt", "mean"),
        std_amt=("amt", "std"),
        unique_customers=("cc_num", "nunique"),
        active_hours=("hour", "nunique")
    )
    .reset_index()
)

daily_test["std_amt"] = (
    daily_test["std_amt"].fillna(0)
)


# ------------------------------------------------------------
# 5. Create TEST labels separately
# ------------------------------------------------------------
# Labels are ONLY used later for evaluation.
# They are NOT used to calculate TEST features.

test_labels = (
    test.groupby(["merchant", "date"])["is_fraud"]
    .max()
    .reset_index()
    .rename(columns={"is_fraud": "had_fraud"})
)


# ------------------------------------------------------------
# 6. Combine TRAIN history + TEST days
# ------------------------------------------------------------

historical = daily_train_test.copy()
historical["source"] = "train"

test_days = daily_test.copy()
test_days["source"] = "test"

combined = pd.concat(
    [historical, test_days],
    ignore_index=True
)

# TRAIN always comes before TEST for the same date.
combined["source_order"] = (
    combined["source"]
    .map({"train": 0, "test": 1})
)

combined = combined.sort_values(
    ["merchant", "date", "source_order"]
).reset_index(drop=True)


# ------------------------------------------------------------
# 7. Create past-only historical statistics
# ------------------------------------------------------------
# shift(1) ensures the CURRENT day is never included.
#
# For TEST:
#
# Day 1 → TRAIN history
# Day 2 → TRAIN + TEST Day 1
# Day 3 → TRAIN + TEST Days 1-2
# etc.

g = combined.groupby("merchant", sort=False)


# Number of previous merchant-days
combined["hist_n"] = g.cumcount()


# -------------------------
# Transaction count
# -------------------------

combined["hist_sum_count"] = (
    g["txn_count"].cumsum()
    .groupby(combined["merchant"])
    .shift(1)
)

combined["hist_sum_count_sq"] = (
    combined["txn_count"] ** 2
)

combined["hist_sum_count_sq"] = (
    combined.groupby("merchant")["hist_sum_count_sq"]
    .cumsum()
    .groupby(combined["merchant"])
    .shift(1)
)


# -------------------------
# Total amount
# -------------------------

combined["hist_sum_total_amt"] = (
    g["total_amt"].cumsum()
    .groupby(combined["merchant"])
    .shift(1)
)

combined["hist_sum_total_amt_sq"] = (
    combined["total_amt"] ** 2
)

combined["hist_sum_total_amt_sq"] = (
    combined.groupby("merchant")["hist_sum_total_amt_sq"]
    .cumsum()
    .groupby(combined["merchant"])
    .shift(1)
)


# -------------------------
# Average amount
# -------------------------

combined["hist_sum_avg_amt"] = (
    g["avg_amt"].cumsum()
    .groupby(combined["merchant"])
    .shift(1)
)

combined["hist_sum_avg_amt_sq"] = (
    combined["avg_amt"] ** 2
)

combined["hist_sum_avg_amt_sq"] = (
    combined.groupby("merchant")["hist_sum_avg_amt_sq"]
    .cumsum()
    .groupby(combined["merchant"])
    .shift(1)
)


# -------------------------
# Amount standard deviation
# -------------------------

combined["hist_sum_std_amt"] = (
    g["std_amt"].cumsum()
    .groupby(combined["merchant"])
    .shift(1)
)

combined["hist_sum_std_amt_sq"] = (
    combined["std_amt"] ** 2
)

combined["hist_sum_std_amt_sq"] = (
    combined.groupby("merchant")["hist_sum_std_amt_sq"]
    .cumsum()
    .groupby(combined["merchant"])
    .shift(1)
)


# -------------------------
# Unique customers
# -------------------------

combined["hist_sum_customers"] = (
    g["unique_customers"].cumsum()
    .groupby(combined["merchant"])
    .shift(1)
)

combined["hist_sum_customers_sq"] = (
    combined["unique_customers"] ** 2
)

combined["hist_sum_customers_sq"] = (
    combined.groupby("merchant")["hist_sum_customers_sq"]
    .cumsum()
    .groupby(combined["merchant"])
    .shift(1)
)


# -------------------------
# Active hours
# -------------------------

combined["hist_sum_hours"] = (
    g["active_hours"].cumsum()
    .groupby(combined["merchant"])
    .shift(1)
)

combined["hist_sum_hours_sq"] = (
    combined["active_hours"] ** 2
)

combined["hist_sum_hours_sq"] = (
    combined.groupby("merchant")["hist_sum_hours_sq"]
    .cumsum()
    .groupby(combined["merchant"])
    .shift(1)
)


# ------------------------------------------------------------
# 8. Historical z-score function
# ------------------------------------------------------------

def historical_z(value, hist_sum, hist_sum_sq, n):

    # Not enough history
    if pd.isna(n) or n < 2:
        return 0.0

    mean = hist_sum / n

    variance = (
        hist_sum_sq / n
    ) - (mean ** 2)

    # Numerical protection
    variance = max(variance, 0)

    std = np.sqrt(variance)

    # Avoid division by zero
    if std == 0 or np.isnan(std):
        std = 1.0

    return (value - mean) / std


# ------------------------------------------------------------
# 9. Calculate six TEST deviation features
# ------------------------------------------------------------

combined["z_count"] = combined.apply(
    lambda r: historical_z(
        r["txn_count"],
        r["hist_sum_count"],
        r["hist_sum_count_sq"],
        r["hist_n"]
    ),
    axis=1
)

combined["z_total_amt"] = combined.apply(
    lambda r: historical_z(
        r["total_amt"],
        r["hist_sum_total_amt"],
        r["hist_sum_total_amt_sq"],
        r["hist_n"]
    ),
    axis=1
)

combined["z_avg_amt"] = combined.apply(
    lambda r: historical_z(
        r["avg_amt"],
        r["hist_sum_avg_amt"],
        r["hist_sum_avg_amt_sq"],
        r["hist_n"]
    ),
    axis=1
)

combined["z_std_amt"] = combined.apply(
    lambda r: historical_z(
        r["std_amt"],
        r["hist_sum_std_amt"],
        r["hist_sum_std_amt_sq"],
        r["hist_n"]
    ),
    axis=1
)

combined["z_unique_customers"] = combined.apply(
    lambda r: historical_z(
        r["unique_customers"],
        r["hist_sum_customers"],
        r["hist_sum_customers_sq"],
        r["hist_n"]
    ),
    axis=1
)

combined["z_active_hours"] = combined.apply(
    lambda r: historical_z(
        r["active_hours"],
        r["hist_sum_hours"],
        r["hist_sum_hours_sq"],
        r["hist_n"]
    ),
    axis=1
)


# ------------------------------------------------------------
# 10. Extract TEST only
# ------------------------------------------------------------

test_model = combined[
    combined["source"] == "test"
].copy()


# ------------------------------------------------------------
# 11. Keep only required columns
# ------------------------------------------------------------

features = [
    "z_count",
    "z_total_amt",
    "z_avg_amt",
    "z_std_amt",
    "z_unique_customers",
    "z_active_hours"
]

test_model = test_model[
    ["merchant", "date"] + features
].copy()


# ------------------------------------------------------------
# 12. Clean numerical values
# ------------------------------------------------------------

test_model[features] = (
    test_model[features]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)


# ------------------------------------------------------------
# 13. Attach TEST labels
# ------------------------------------------------------------

test_model = test_model.merge(
    test_labels,
    on=["merchant", "date"],
    how="left"
)


# ------------------------------------------------------------
# 14. Final verification
# ------------------------------------------------------------

print("======================================")
print("STRICT TEST DATASET")
print("======================================")

print("\nTEST merchant-days:")
print(len(test_model))

print("\nExpected TEST merchant-days:")
print(len(daily_test))

print("\nFeatures:")
print(features)

print("\nFraud-day distribution:")
print(
    test_model["had_fraud"].value_counts()
)

print("\nFraud-day percentage:")
print(
    f"{test_model['had_fraud'].mean() * 100:.2f}%"
)

print("\nAverage deviations by fraud label:")

display(
    test_model.groupby("had_fraud")[features].mean()
)

print("\nSample:")

display(
    test_model[
        ["merchant", "date", "had_fraud"] + features
    ].head(10)
)

## Step 7 — Final evaluation on held-out test data
This is the headline result: precision, recall, ROC-AUC, PR-AUC and the confusion matrix on merchant-days the model never trained on, at a threshold selected using validation data only (never test data) — an honest, non-cherry-picked number.

In [ ]:
# ============================================================
# PHASE 14 — FINAL TEST EVALUATION
# ============================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)

# ------------------------------------------------------------
# 1. Prepare TEST features and labels
# ------------------------------------------------------------

X_test = test_model[features]
y_test = test_model["had_fraud"]


# ------------------------------------------------------------
# 2. Generate fraud probabilities
# ------------------------------------------------------------

test_probability = model_strict.predict_proba(
    X_test
)[:, 1]


# ------------------------------------------------------------
# 3. LOCKED THRESHOLD
# ------------------------------------------------------------
# Selected using validation data only.

FINAL_THRESHOLD = 0.75

test_prediction = (
    test_probability >= FINAL_THRESHOLD
).astype(int)


# ------------------------------------------------------------
# 4. Calculate metrics
# ------------------------------------------------------------

precision = precision_score(
    y_test,
    test_prediction,
    zero_division=0
)

recall = recall_score(
    y_test,
    test_prediction,
    zero_division=0
)

f1 = f1_score(
    y_test,
    test_prediction,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test,
    test_probability
)

pr_auc = average_precision_score(
    y_test,
    test_probability
)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    test_prediction
).ravel()

alert_rate = (
    (fp + tp) / len(y_test)
)


# ------------------------------------------------------------
# 5. Final results
# ------------------------------------------------------------

print("======================================")
print("FINAL TEST RESULTS")
print("======================================")

print(f"Threshold : {FINAL_THRESHOLD:.2f}")

print(f"\nPrecision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")
print(f"PR-AUC    : {pr_auc:.4f}")

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        test_prediction
    )
)

print("\nDetailed counts:")
print(f"True Negatives  : {tn}")
print(f"False Positives : {fp}")
print(f"False Negatives : {fn}")
print(f"True Positives  : {tp}")

print(f"\nAlert rate: {alert_rate:.4f}")
print(f"Alert percentage: {alert_rate * 100:.2f}%")

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        test_prediction,
        digits=4,
        zero_division=0
    )
)


## Step 8 — Convert model probability into a risk score and risk level
A 0–100 risk score is more interpretable for a judge or a merchant-ops reviewer than a raw probability. Bucket thresholds (Low/Medium/High/Critical) are a simple, named heuristic — not statistically derived — and that should be stated plainly if asked.

In [ ]:
# ============================================================
# PHASE 15 — RISK SCORE + RISK LEVEL
# ============================================================

# ------------------------------------------------------------
# 1. Convert model probability into a 0-100 risk score
# ------------------------------------------------------------

test_model["risk_probability"] = test_probability

test_model["risk_score"] = (
    test_model["risk_probability"] * 100
).round(2)


# ------------------------------------------------------------
# 2. Assign risk levels
# ------------------------------------------------------------

def assign_risk_level(score):

    if score < 30:
        return "Low"

    elif score < 60:
        return "Medium"

    elif score < 80:
        return "High"

    else:
        return "Critical"


test_model["risk_level"] = (
    test_model["risk_score"]
    .apply(assign_risk_level)
)


# ------------------------------------------------------------
# 3. Final alert decision
# ------------------------------------------------------------

FINAL_THRESHOLD = 0.75

test_model["is_alert"] = (
    test_model["risk_probability"]
    >= FINAL_THRESHOLD
)


# ------------------------------------------------------------
# 4. Check distribution
# ------------------------------------------------------------

print("======================================")
print("RISK SCORE DISTRIBUTION")
print("======================================")

print("\nRisk level distribution:")

print(
    test_model["risk_level"]
    .value_counts()
)


print("\nAlert distribution:")

print(
    test_model["is_alert"]
    .value_counts()
)


print("\nRisk score statistics:")

print(
    test_model["risk_score"].describe()
)


# ------------------------------------------------------------
# 5. Show highest-risk merchant-days
# ------------------------------------------------------------

print("\nHighest-risk merchant-days:")

display(
    test_model[
        [
            "merchant",
            "date",
            "risk_score",
            "risk_level",
            "is_alert",
            "had_fraud"
        ]
        + features
    ]
    .sort_values(
        "risk_score",
        ascending=False
    )
    .head(20)
)

## Step 9 — Human-readable risk explanations
For any flagged merchant-day, this generates the top reasons it was flagged, directly from the deviation features that exceeded a z-score threshold — not from the model's internal weights, so it's honest about *what* triggered the alert, in plain language.

In [ ]:
# ============================================================
# PHASE 16 — HUMAN-READABLE RISK EXPLANATIONS
# ============================================================

def generate_reasons(row):

    reasons = []

    # --------------------------------------------------------
    # Total transaction amount
    # --------------------------------------------------------

    if row["z_total_amt"] >= 2:
        reasons.append(
            "Transaction amount is significantly above the merchant's normal level"
        )

    elif row["z_total_amt"] >= 1:
        reasons.append(
            "Transaction amount is above the merchant's normal level"
        )


    # --------------------------------------------------------
    # Average transaction amount
    # --------------------------------------------------------

    if row["z_avg_amt"] >= 2:
        reasons.append(
            "Average transaction size is significantly unusual"
        )

    elif row["z_avg_amt"] >= 1:
        reasons.append(
            "Average transaction size is unusually high"
        )


    # --------------------------------------------------------
    # Amount variability
    # --------------------------------------------------------

    if row["z_std_amt"] >= 2:
        reasons.append(
            "Transaction amount variability is significantly unusual"
        )

    elif row["z_std_amt"] >= 1:
        reasons.append(
            "Transaction amount variability is unusually high"
        )


    # --------------------------------------------------------
    # Transaction count
    # --------------------------------------------------------

    if row["z_count"] >= 2:
        reasons.append(
            "Transaction volume is significantly above normal"
        )

    elif row["z_count"] >= 1:
        reasons.append(
            "Transaction volume is above normal"
        )


    # --------------------------------------------------------
    # Customer activity
    # --------------------------------------------------------

    if row["z_unique_customers"] >= 2:
        reasons.append(
            "Number of unique customers is significantly unusual"
        )

    elif row["z_unique_customers"] >= 1:
        reasons.append(
            "Number of unique customers is above normal"
        )


    # --------------------------------------------------------
    # Active hours
    # --------------------------------------------------------

    if row["z_active_hours"] >= 2:
        reasons.append(
            "Merchant activity spans significantly more hours than usual"
        )

    elif row["z_active_hours"] >= 1:
        reasons.append(
            "Merchant activity spans more hours than usual"
        )


    # --------------------------------------------------------
    # If no strong reason
    # --------------------------------------------------------

    if len(reasons) == 0:
        reasons.append(
            "Overall transaction behavior differs from the merchant's historical pattern"
        )


    # Keep maximum 3 reasons
    return reasons[:3]


test_model["risk_reasons"] = (
    test_model.apply(
        generate_reasons,
        axis=1
    )
)


# ------------------------------------------------------------
# Convert reasons to dashboard-friendly text
# ------------------------------------------------------------

test_model["risk_reason_text"] = (
    test_model["risk_reasons"]
    .apply(lambda x: " • ".join(x))
)


# ------------------------------------------------------------
# Show examples
# ------------------------------------------------------------

print("Example risk explanations:")

display(
    test_model[
        [
            "merchant",
            "date",
            "risk_score",
            "risk_level",
            "risk_reason_text",
            "had_fraud"
        ]
    ]
    .sort_values(
        "risk_score",
        ascending=False
    )
    .head(15)
)

## Step 10 — Feature importance
Which deviation signals the model actually relies on most — useful both for explainability and for sanity-checking that the model learned something sensible.

In [ ]:
# ============================================================
# PHASE 17 — FEATURE IMPORTANCE + SIGNAL ANALYSIS
# ============================================================

# ------------------------------------------------------------
# 1. XGBoost feature importance
# ------------------------------------------------------------

importance_df = pd.DataFrame({
    "feature": features,
    "importance": model_strict.feature_importances_
})

importance_df = (
    importance_df
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

print("XGBoost feature importance:")

display(
    importance_df
)


# ------------------------------------------------------------
# 2. Absolute deviation by fraud label
# ------------------------------------------------------------

test_signal = test_model.copy()

for feature in features:

    test_signal[
        feature + "_abs"
    ] = test_signal[feature].abs()


abs_features = [
    feature + "_abs"
    for feature in features
]

print("\nAbsolute deviation by fraud label:")

display(
    test_signal.groupby("had_fraud")[
        abs_features
    ].mean()
)

## Step 11 — Per-merchant risk summary
Aggregates alerts and detection rate per merchant across the whole test period.

In [ ]:
# ============================================================
# PHASE 18 — MERCHANT RISK SUMMARY
# ============================================================

merchant_summary = (
    test_model
    .groupby("merchant")
    .agg(
        monitored_days=("date", "count"),

        alerts=("is_alert", "sum"),

        alert_rate=("is_alert", "mean"),

        average_risk=("risk_score", "mean"),

        maximum_risk=("risk_score", "max"),

        fraud_days=("had_fraud", "sum"),

        detected_fraud_days=(
            "is_alert",
            lambda x: (
                x
                & (
                    test_model.loc[
                        x.index,
                        "had_fraud"
                    ].astype(bool)
                )
            ).sum()
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# Merchant alert rate
# ------------------------------------------------------------

merchant_summary["alert_rate"] = (
    merchant_summary["alert_rate"] * 100
).round(2)


# ------------------------------------------------------------
# Detection rate for merchants that experienced fraud
# ------------------------------------------------------------

merchant_summary["fraud_detection_rate"] = np.where(
    merchant_summary["fraud_days"] > 0,

    (
        merchant_summary["detected_fraud_days"]
        / merchant_summary["fraud_days"]
        * 100
    ).round(2),

    np.nan
)


# ------------------------------------------------------------
# Sort by maximum risk
# ------------------------------------------------------------

merchant_summary = merchant_summary.sort_values(
    [
        "maximum_risk",
        "alerts"
    ],
    ascending=False
).reset_index(drop=True)


print("Merchant summary:")

display(
    merchant_summary.head(20)
)

## Step 12 — Alert queue
The actual output a risk-ops team would work from: ranked, highest-risk merchant-days first, with reasons attached.

In [ ]:
# ============================================================
# PHASE 19 — RISK ALERT QUEUE
# ============================================================

alert_queue = test_model[
    test_model["is_alert"] == True
].copy()


# ------------------------------------------------------------
# Add useful display fields
# ------------------------------------------------------------

alert_queue["risk_score"] = (
    alert_queue["risk_score"].round(1)
)

alert_queue["risk_probability"] = (
    alert_queue["risk_probability"].round(4)
)

alert_queue["date"] = pd.to_datetime(
    alert_queue["date"]
)


# ------------------------------------------------------------
# Sort highest risk first
# ------------------------------------------------------------

alert_queue = alert_queue.sort_values(
    [
        "risk_score",
        "z_std_amt",
        "z_total_amt"
    ],
    ascending=False
).reset_index(drop=True)


# ------------------------------------------------------------
# Dashboard columns
# ------------------------------------------------------------

alert_queue = alert_queue[
    [
        "merchant",
        "date",
        "risk_score",
        "risk_level",
        "risk_reason_text",

        "z_count",
        "z_total_amt",
        "z_avg_amt",
        "z_std_amt",
        "z_unique_customers",
        "z_active_hours",

        "had_fraud"
    ]
]


print("Total alerts:", len(alert_queue))

print("\nAlert queue:")

display(
    alert_queue.head(25)
)

## Step 13 — Final model report
A single summary object capturing every key number for the writeup and the interview.

In [ ]:
# ============================================================
# PHASE 20 — FINAL MODEL REPORT
# ============================================================

final_report = {
    "model": "XGBoost",

    "problem": "Merchant-day fraud-spike detection",

    "dataset": "Kartik2112 simulated credit-card transaction dataset",

    "training_merchant_days": len(X_train),

    "validation_merchant_days": len(X_val),

    "test_merchant_days": len(X_test),

    "test_fraud_days": int(y_test.sum()),

    "test_fraud_rate_percent": round(
        y_test.mean() * 100,
        2
    ),

    "threshold": FINAL_THRESHOLD,

    "precision": round(
        precision,
        4
    ),

    "recall": round(
        recall,
        4
    ),

    "f1": round(
        f1,
        4
    ),

    "roc_auc": round(
        roc_auc,
        4
    ),

    "pr_auc": round(
        pr_auc,
        4
    ),

    "true_negatives": int(tn),

    "false_positives": int(fp),

    "false_negatives": int(fn),

    "true_positives": int(tp),

    "alert_rate_percent": round(
        alert_rate * 100,
        2
    )
}


print("======================================")
print("FINAL PROJECT REPORT")
print("======================================")

for key, value in final_report.items():

    print(
        f"{key}: {value}"
    )

## Step 14 — Save dashboard-ready outputs

In [ ]:
# ============================================================
# PHASE 21 — SAVE DASHBOARD-READY DATA
# ============================================================

# ------------------------------------------------------------
# 1. Save daily risk data
# ------------------------------------------------------------

test_model.to_csv(
    "merchant_daily_risk.csv",
    index=False
)


# ------------------------------------------------------------
# 2. Save alert queue
# ------------------------------------------------------------

alert_queue.to_csv(
    "risk_alert_queue.csv",
    index=False
)


# ------------------------------------------------------------
# 3. Save merchant summary
# ------------------------------------------------------------

merchant_summary.to_csv(
    "merchant_risk_summary.csv",
    index=False
)


# ------------------------------------------------------------
# 4. Save final metrics
# ------------------------------------------------------------

pd.DataFrame(
    [final_report]
).to_csv(
    "final_model_metrics.csv",
    index=False
)


# ------------------------------------------------------------
# 5. Confirmation
# ------------------------------------------------------------

print("======================================")
print("FILES CREATED")
print("======================================")

print("✓ merchant_daily_risk.csv")
print("✓ risk_alert_queue.csv")
print("✓ merchant_risk_summary.csv")
print("✓ final_model_metrics.csv")

---
## Known limitations (state these proactively — it's a strength, not a weakness)

- **Dataset is simulated (Sparkov), not real payment data** — absolute numbers won't transfer directly to production, but the *methodology* (merchant-relative walk-forward baselines) does.
- **Risk-level buckets (Low/Medium/High/Critical) are a fixed heuristic**, not statistically calibrated.
- **No dormancy/reactivation modeling** — the baseline is built only from each merchant's active days.
- **Isolation Forest was tested and found weaker than the supervised model** on this labeled dataset.
- **Precision at the chosen threshold is modest** — expected given ~1.7% fraud-day prevalence; this is why we report the full threshold/cost table rather than one number.
